In [10]:
"""
signal_decomposition.py
=======================
Produces Table 3 and all numbers cited in Section 5.3

Inputs:
  rsun_station_values.csv
  surfrad_processed_0.30/surfrad_multiyear_means.csv

Outputs:
  validation_outputs_0.30/table3_partA.csv
  validation_outputs_0.30/table3_partB.csv
  validation_outputs_0.30/table3_partC.csv
  Console: all inline numbers cited in Section 5.3
"""

import os
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
BASE         = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis"
SURFRAD_FILE = os.path.join(BASE, "surfrad_processed_0.30", "surfrad_multiyear_means.csv")
RSUN_FILE    = os.path.join(BASE, "rsun_station_values.csv")
OUT_DIR      = os.path.join(BASE, "validation_outputs_final", "0.30")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

STATION_NAMES = {
    "gwn": "Goodwin Creek, MS",
    "dra": "Desert Rock, NV",
    "psu": "Penn State, PA",
    "bon": "Bondville, IL",
    "tbl": "Table Mountain, CO",
    "sxf": "Sioux Falls, SD",
    "fpk": "Fort Peck, MT",
}
STATION_ARIDITY = {
    "dra": "Arid",
    "tbl": "Semi-arid montane",
    "sxf": "Semi-arid",
    "fpk": "Semi-arid",
    "psu": "Humid",
    "bon": "Humid",
    "gwn": "Humid subtropical",
}
ARIDITY_ORDER = ["Arid","Semi-arid montane","Semi-arid","Humid","Humid subtropical"]

DOY_MONTH = {
    15:"Jan", 45:"Feb", 74:"Mar", 105:"Apr",
    135:"May", 166:"Jun", 196:"Jul", 227:"Aug",
    258:"Sep", 288:"Oct", 319:"Nov", 349:"Dec",
}
DOY_SEASON = {
    15:"Winter", 45:"Winter",
    74:"Spring/Autumn", 105:"Spring/Autumn",
    135:"Summer", 166:"Summer",
    196:"Summer", 227:"Summer",
    258:"Spring/Autumn", 288:"Spring/Autumn",
    319:"Winter", 349:"Winter",
}

# ============================================================
# 1. LOAD AND JOIN — glob_rad only (signal decomposition uses glob_rad)
# ============================================================
print("Loading data...")

rsun = pd.read_csv(RSUN_FILE)
rsun = rsun[rsun["doy"] != "annual"].copy()
rsun["doy"]       = rsun["doy"].astype(int)
rsun["rsun_Whm2"] = pd.to_numeric(rsun["rsun_Whm2"], errors="coerce")
rsun = rsun[rsun["component"] == "glob_rad"][["station","doy","component","rsun_Whm2"]].copy()

surfrad = pd.read_csv(SURFRAD_FILE)
surfrad["doy"] = surfrad["doy"].astype(int)

surf_long = surfrad.melt(
    id_vars=["station","doy","lat"],
    value_vars=["glob_rad_mean"],
    var_name="component", value_name="surfrad_Whm2"
)
surf_long["component"] = "glob_rad"

val = rsun.merge(surf_long, on=["station","doy","component"], how="inner").dropna()
print(f"Joined: {len(val)} rows | "
      f"{val['station'].nunique()} stations | "
      f"{val['doy'].nunique()} DOYs")


Loading data...
Joined: 84 rows | 7 stations | 12 DOYs


In [11]:
# ============================================================
# 2. PART A — Network-level R² at three aggregation levels
# ============================================================
doy_means = val.groupby("doy")[["rsun_Whm2","surfrad_Whm2"]].mean()
sta_means = val.groupby("station")[["rsun_Whm2","surfrad_Whm2"]].mean()

r2_seasonal = np.corrcoef(doy_means["rsun_Whm2"], doy_means["surfrad_Whm2"])[0,1]**2
r2_spatial  = np.corrcoef(sta_means["rsun_Whm2"],  sta_means["surfrad_Whm2"])[0,1]**2
r2_overall  = np.corrcoef(val["rsun_Whm2"],        val["surfrad_Whm2"])[0,1]**2

part_a = pd.DataFrame([
    {"Signal"      : "Seasonal",
     "Aggregation" : "DOY means, 7-station average (n=12)",
     "R2"          : round(r2_seasonal, 3),
     "Interpretation": "Solar declination cycle captured with high consistency"},
    {"Signal"      : "Spatial",
     "Aggregation" : "Station means, 12-DOY average (n=7)",
     "R2"          : round(r2_spatial, 3),
     "Interpretation": "Latitudinal gradient reproduced; humid-station turbidity adds noise"},
    {"Signal"      : "Overall",
     "Aggregation" : "All station-DOY pairs (n=84)",
     "R2"          : round(r2_overall, 3),
     "Interpretation": "Combined seasonal and spatial signal"},
])


In [12]:
# ============================================================
# 3. PART B — Within-station seasonal R² by climate regime
# ============================================================
part_b_rows = []
for st in val["station"].unique():
    sub = val[val["station"] == st]
    if len(sub) < 3:
        continue
    r2 = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0,1]**2
    part_b_rows.append({
        "Station"  : STATION_NAMES.get(st, st),
        "Aridity"  : STATION_ARIDITY.get(st, "Unknown"),
        "R2"       : round(r2, 3),
        "n_DOYs"   : len(sub),
    })

part_b = pd.DataFrame(part_b_rows)
part_b["aridity_rank"] = part_b["Aridity"].map(
    {a: i for i, a in enumerate(ARIDITY_ORDER)}
)
part_b = part_b.sort_values("aridity_rank").drop("aridity_rank", axis=1).reset_index(drop=True)


In [13]:
# ============================================================
# 4. PART C — Within-DOY spatial R² by season
# ============================================================
part_c_rows = []
for doy in sorted(val["doy"].unique()):
    sub = val[val["doy"] == doy]
    if len(sub) < 3:
        continue
    r2 = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0,1]**2
    part_c_rows.append({
        "DOY"    : doy,
        "Month"  : DOY_MONTH.get(doy, ""),
        "Season" : DOY_SEASON.get(doy, "Unknown"),
        "R2"     : round(r2, 3),
        "n_sta"  : len(sub),
    })

part_c = pd.DataFrame(part_c_rows)

In [14]:
# ============================================================
# 5. PRINT TABLE 3
# ============================================================
print("\n" + "="*70)
print(" TABLE 3 — Signal decomposition")
print("="*70)

print("\nPart A — Network-level R² (glob_rad)")
print(f"  {'Signal':<12} {'R2':>6}  Aggregation")
print("-"*65)
for _, r in part_a.iterrows():
    print(f"  {r['Signal']:<12} {r['R2']:>6.3f}  {r['Aggregation']}")

print("\nPart B — Within-station seasonal R² by aridity (glob_rad, n=12 DOYs)")
print(f"  {'Station':<24} {'Aridity':<22} {'R2':>6}")
print("-"*60)
for _, r in part_b.iterrows():
    print(f"  {r['Station']:<24} {r['Aridity']:<22} {r['R2']:>6.3f}")

print("\nPart C — Within-DOY spatial R² by season (glob_rad, n=7 stations)")
print(f"  {'DOY':>5} {'Month':>5} {'Season':<16} {'R2':>6}")
print("-"*40)
for _, r in part_c.iterrows():
    print(f"  {int(r['DOY']):>5} {r['Month']:>5} {r['Season']:<16} {r['R2']:>6.3f}")



 TABLE 3 — Signal decomposition

Part A — Network-level R² (glob_rad)
  Signal           R2  Aggregation
-----------------------------------------------------------------
  Seasonal      0.955  DOY means, 7-station average (n=12)
  Spatial       0.435  Station means, 12-DOY average (n=7)
  Overall       0.596  All station-DOY pairs (n=84)

Part B — Within-station seasonal R² by aridity (glob_rad, n=12 DOYs)
  Station                  Aridity                    R2
------------------------------------------------------------
  Desert Rock, NV          Arid                    0.862
  Table Mountain, CO       Semi-arid montane       0.781
  Fort Peck, MT            Semi-arid               0.796
  Sioux Falls, SD          Semi-arid               0.833
  Bondville, IL            Humid                   0.515
  Penn State, PA           Humid                   0.588
  Goodwin Creek, MS        Humid subtropical       0.300

Part C — Within-DOY spatial R² by season (glob_rad, n=7 stations)
    

In [15]:
# ============================================================
# 6. INLINE NUMBERS — Section 5.3
# ============================================================
print("\n" + "="*70)
print(" INLINE NUMBERS — Section 5.3")
print("="*70)
print(f"\n  Seasonal R2 (DOY means)  = {r2_seasonal:.3f}")
print(f"  Spatial R2  (sta means)  = {r2_spatial:.3f}")
print(f"  Overall R2  (all pairs)  = {r2_overall:.3f}")

print("\n  Winter DOYs spatial R2:")
for _, r in part_c[part_c["Season"]=="Winter"].iterrows():
    print(f"    DOY {int(r['DOY']):03d} ({r['Month']}): R2 = {r['R2']:.3f}")

print("\n  Summer DOYs spatial R2:")
for _, r in part_c[part_c["Season"]=="Summer"].iterrows():
    print(f"    DOY {int(r['DOY']):03d} ({r['Month']}): R2 = {r['R2']:.3f}")

print("\n  Spring/Autumn DOYs spatial R2:")
for _, r in part_c[part_c["Season"]=="Spring/Autumn"].iterrows():
    print(f"    DOY {int(r['DOY']):03d} ({r['Month']}): R2 = {r['R2']:.3f}")

print("\n  Seasonal range (glob_rad):")
doy_rsun_mean    = val.groupby("doy")["rsun_Whm2"].mean()
doy_surfrad_mean = val.groupby("doy")["surfrad_Whm2"].mean()
print(f"    r.sun   : {doy_rsun_mean.max()-doy_rsun_mean.min():.0f} Wh/m2/day")
print(f"    SURFRAD : {doy_surfrad_mean.max()-doy_surfrad_mean.min():.0f} Wh/m2/day")

print("\n  Spatial range (glob_rad):")
sta_rsun_mean    = val.groupby("station")["rsun_Whm2"].mean()
sta_surfrad_mean = val.groupby("station")["surfrad_Whm2"].mean()
print(f"    r.sun   : {sta_rsun_mean.max()-sta_rsun_mean.min():.0f} Wh/m2/day")
print(f"    SURFRAD : {sta_surfrad_mean.max()-sta_surfrad_mean.min():.0f} Wh/m2/day")



 INLINE NUMBERS — Section 5.3

  Seasonal R2 (DOY means)  = 0.955
  Spatial R2  (sta means)  = 0.435
  Overall R2  (all pairs)  = 0.596

  Winter DOYs spatial R2:
    DOY 015 (Jan): R2 = 0.712
    DOY 045 (Feb): R2 = 0.119
    DOY 319 (Nov): R2 = 0.728
    DOY 349 (Dec): R2 = 0.384

  Summer DOYs spatial R2:
    DOY 135 (May): R2 = 0.074
    DOY 166 (Jun): R2 = 0.072
    DOY 196 (Jul): R2 = 0.009
    DOY 227 (Aug): R2 = 0.097

  Spring/Autumn DOYs spatial R2:
    DOY 074 (Mar): R2 = 0.503
    DOY 105 (Apr): R2 = 0.515
    DOY 258 (Sep): R2 = 0.158
    DOY 288 (Oct): R2 = 0.437

  Seasonal range (glob_rad):
    r.sun   : 6493 Wh/m2/day
    SURFRAD : 3433 Wh/m2/day

  Spatial range (glob_rad):
    r.sun   : 1364 Wh/m2/day
    SURFRAD : 1958 Wh/m2/day


In [16]:
# ============================================================
# 7. SAVE
# ============================================================
part_a.to_csv(os.path.join(OUT_DIR, "table3_partA.csv"), index=False)
part_b.to_csv(os.path.join(OUT_DIR, "table3_partB.csv"), index=False)
part_c.to_csv(os.path.join(OUT_DIR, "table3_partC.csv"), index=False)

print(f"\nSaved:")
print(f"  table3_partA.csv")
print(f"  table3_partB.csv")
print(f"  table3_partC.csv")
print(f"  All outputs in: {OUT_DIR}")


Saved:
  table3_partA.csv
  table3_partB.csv
  table3_partC.csv
  All outputs in: C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\validation_outputs_final\0.30
